# Prior Check via PQMass

Validate the trained 256x256 score-based **prior** (`outputs/probes_final`, EMA)
by drawing *unconditional* samples `x ~ p(x)` and testing whether they are
statistically indistinguishable from the real PROBES galaxies, using
[PQMass](https://github.com/Ciela-Institute/PQM) (a Voronoi-tessellation
two-sample chi^2 test).

Workflow: a small **timed trial** here to gauge per-sample cost (then promote the
full run to `sbatch run_sample_prior.sh` if needed), followed by the PQMass
analysis in **pixel space** (primary) and a **PCA-reduced** cross-check.

## 1. Setup

In [10]:
import sys, time
from pathlib import Path

# Locate the repo root (dir containing src/sample.py) regardless of the kernel's
# working directory, falling back to the known absolute path on Wilkes3.
_candidates = [Path.cwd(), *Path.cwd().parents,
               Path("/rds/user/yd388/hpc-work/DIS-Project-Lensed-Galaxy")]
REPO = next(p for p in _candidates if (p / "src" / "sample.py").exists())
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import chi2 as chi2_dist

from sample import load_model, to_display_flux, atomic_save
from train_prior import load_probes
from pqm import pqm_chi2, pqm_pvalue

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}  |  repo: {REPO}")

OUTPUT_DIR = REPO / "outputs" / "probes_final"
DATA_DIR = REPO / "data" / "gals_gband_norm"
PRIOR_DIR = OUTPUT_DIR / "prior_check"
PRIOR_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 256

Using device: cuda  |  repo: /rds/user/yd388/hpc-work/DIS-Project-Lensed-Galaxy


## 2. Load the trained prior (EMA)

In [11]:
model, step = load_model(OUTPUT_DIR / "latest.pt", device)
print(f"Prior loaded at step {step}")

Using the Variance Exploding SDE
Loaded EMA at step 350980, sigma_max=263.4, sigma_min=0.0001
Prior loaded at step 350980


## 3. Timed trial sampling

Draw a small batch interactively and project the wall-time for the full set.
If the projection is large, run the full generation as a batch job instead:
`sbatch run_sample_prior.sh` (writes `prior_check/prior_samples.pt`).

In [12]:
N_TRIAL = 32     # small interactive batch
STEPS   = 2000   # Euler-Maruyama steps. Quality is very sensitive to this for VE
                 # (sigma_max=263); 1000 leaves soft/half-denoised samples. The 128
                 # prior used 4000 and the posterior sampler uses 8000 -- raise if
                 # samples still look hazy.
N_FULL  = 1000   # target sample count for the real check

torch.manual_seed(21)
if device.type == "cuda":
    torch.cuda.manual_seed_all(21)

t0 = time.time()
trial = model.sample(shape=[N_TRIAL, 1, IMAGE_SIZE, IMAGE_SIZE], steps=STEPS).cpu()
dt = time.time() - t0

per_sample = dt / N_TRIAL
print(f"{N_TRIAL} samples in {dt:.1f}s  ->  {per_sample:.2f}s/sample")
print(f"Projected for N_FULL={N_FULL}: {per_sample * N_FULL / 60:.1f} min")
print(f"trial: {tuple(trial.shape)}, range=[{trial.min():.3f}, {trial.max():.3f}]")

atomic_save(trial, PRIOR_DIR / "prior_samples_interactive.pt")

Sampling from the prior | t = 0.8 | sigma = 2.3e+01| scale ~ 2.3e+01:  16%|███████████▋                                                           | 658/4000 [05:05<25:52,  2.15it/s]


KeyboardInterrupt: 

## 4. Load prior samples + real PROBES data

Prefer the full batch `prior_samples.pt` if it exists, else the interactive
trial. Match the two set sizes so neither dominates the tessellation, then
flatten to `(N, 65536)` for PQMass.

In [ ]:
batch_path = PRIOR_DIR / "prior_samples.pt"
trial_path = PRIOR_DIR / "prior_samples_interactive.pt"
prior_path = batch_path if batch_path.exists() else trial_path
prior = torch.load(prior_path, map_location="cpu")          # (N, 1, H, W)
prior = prior.clamp(-1.0, 1.0)   # match the data domain: PROBES is hard-clipped to
                                 # [-1,1] in preprocessing, but the continuous sampler
                                 # overshoots (~-1.04/1.07); that boundary mismatch alone
                                 # roughly doubles the pixel-space chi^2.
print(f"Loaded {prior.shape[0]} prior samples from {prior_path.name}")

real = load_probes(str(DATA_DIR), n_subset=None, image_size=IMAGE_SIZE)  # (M, H, W)
real = torch.from_numpy(real).unsqueeze(1)                   # (M, 1, H, W)

n = min(prior.shape[0], real.shape[0])
g = torch.Generator().manual_seed(21)
real = real[torch.randperm(real.shape[0], generator=g)[:n]]
prior = prior[:n]
print(f"Using {n} samples per set")

x_real = real.reshape(n, -1).numpy().astype(np.float32)
x_prior = prior.reshape(n, -1).numpy().astype(np.float32)

## 5. Visual sanity check

Eyeball real vs prior samples (log-flux, repo convention) and compare a couple
of per-image summary statistics before trusting the formal test.

In [ ]:
sel = np.random.default_rng(0).choice(n, size=6, replace=False)  # representative, not first-6
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
flux_kw = dict(cmap="magma", norm=LogNorm(vmin=1e-2, vmax=5.5), origin="lower")
for col, j in enumerate(sel):
    axes[0, col].imshow(to_display_flux(real[j, 0]), **flux_kw)
    axes[1, col].imshow(to_display_flux(prior[j, 0]), **flux_kw)
    for i in range(2):
        axes[i, col].axis("off")
axes[0, 0].set_title("real PROBES", loc="left")
axes[1, 0].set_title("prior samples", loc="left")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for k, (name, fn) in enumerate([("total flux", lambda a: a.sum(1)),
                                ("peak intensity", lambda a: a.max(1))]):
    ax[k].hist(fn(x_real), bins=40, density=True, alpha=0.6, label="real")
    ax[k].hist(fn(x_prior), bins=40, density=True, alpha=0.6, label="prior")
    ax[k].set_title(name); ax[k].legend()
plt.tight_layout(); plt.show()

## 6. PQMass - pixel space (primary)

`pqm_chi2` / `pqm_pvalue` over many re-tessellations. If prior and data match:
`chi^2 / dof ~ 1` and the p-values are ~Uniform(0, 1) (mean ~ 0.5).

In [ ]:
RE_TESS = 200
# num_refs must be < the samples per set, ideally << it. The real-vs-real sanity
# below uses n//2 per set, so derive num_refs from n to stay safe on a small trial;
# it reaches the full 100 once n >= ~800. A tiny trial is under-powered -- the full
# 1000-sample run is the real test.
NUM_REFS = min(100, max(2, n // 8))
KW = dict(num_refs=NUM_REFS, re_tessellation=RE_TESS, z_score_norm=True, gauss_frac=0.05)
dof = NUM_REFS - 1
print(f"n={n} per set -> num_refs={NUM_REFS}, dof={dof}"
      + ("   (small sample: under-powered, generate the full set)" if NUM_REFS < 100 else ""))

chi2_vals = np.asarray(pqm_chi2(x_prior, x_real, **KW))
pvals = np.asarray(pqm_pvalue(x_prior, x_real, **KW))
print(f"pixel space:  chi2/dof = {chi2_vals.mean() / dof:.3f}  (target ~1)")
print(f"              p-value mean = {pvals.mean():.3f}  (target ~0.5)")

# Sanity: real vs real (disjoint halves) must not be flagged as different.
half = n // 2
p_rr = np.asarray(pqm_pvalue(x_real[:half], x_real[half:2 * half], **KW))
print(f"sanity (real vs real): p-value mean = {p_rr.mean():.3f}  (should be ~0.5)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(chi2_vals, bins=30, density=True, alpha=0.6, label="PQMass")
xs = np.linspace(chi2_vals.min(), chi2_vals.max(), 200)
ax[0].plot(xs, chi2_dist.pdf(xs, dof), "k--", label=f"chi2(dof={dof})")
ax[0].set_title("chi^2 (prior vs real)"); ax[0].legend()
ax[1].hist(pvals, bins=20, range=(0, 1), density=True, alpha=0.6, label="prior vs real")
ax[1].hist(p_rr, bins=20, range=(0, 1), density=True, alpha=0.4, label="real vs real")
ax[1].axhline(1.0, color="k", ls="--", label="uniform")
ax[1].set_title("p-values"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. PQMass - PCA cross-check

65,536 pixel dims is low-power for nearest-reference tessellation. Project both
sets onto the top components of the *combined* data (so the basis is neutral)
and rerun. Agreement with the pixel-space verdict is reassuring.

In [ ]:
N_COMPONENTS = 50
combined = np.concatenate([x_real, x_prior], axis=0)
mean = combined.mean(0, keepdims=True)
# Economy SVD of the centered stack; rows of Vt are the principal directions.
_, _, Vt = np.linalg.svd(combined - mean, full_matrices=False)
comps = Vt[:N_COMPONENTS]                       # (k, D)
z_real = (x_real - mean) @ comps.T              # (n, k)
z_prior = (x_prior - mean) @ comps.T

chi2_pca = np.asarray(pqm_chi2(z_prior, z_real, **KW))
p_pca = np.asarray(pqm_pvalue(z_prior, z_real, **KW))
print(f"PCA({N_COMPONENTS}):  chi2/dof = {chi2_pca.mean() / dof:.3f}   p-value mean = {p_pca.mean():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(chi2_pca, bins=30, density=True, alpha=0.6, label="PQMass (PCA)")
xs = np.linspace(chi2_pca.min(), chi2_pca.max(), 200)
ax[0].plot(xs, chi2_dist.pdf(xs, dof), "k--", label=f"chi2(dof={dof})")
ax[0].set_title(f"chi^2 in PCA({N_COMPONENTS}) space"); ax[0].legend()
ax[1].hist(p_pca, bins=20, range=(0, 1), density=True, alpha=0.6)
ax[1].axhline(1.0, color="k", ls="--", label="uniform")
ax[1].set_title("p-values (PCA)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Interpretation

- **chi^2 / dof ~ 1** and **p-values ~ Uniform(0, 1)** (mean ~ 0.5): the prior is
  statistically indistinguishable from the data over the regions probed.
- **chi^2 / dof >> 1**, **p-values piled near 0**: the prior and data differ
  (mode collapse, wrong support, blurry/sharp mismatch).
- **chi^2 near 0 / spurious p-values**: too few samples for the dimensionality, or
  duplicated/memorized samples. Increase the sample count (`N_FULL`) and/or
  `re_tessellation` and re-check.

The pixel-space and PCA cross-check should tell the same story; a disagreement
usually means the pixel-space test is under-powered at the current sample size.